In [11]:
!pip install dotenv pandas

In [12]:
import os
import time
import smtplib
import pandas as pd
from pathlib import Path
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.image import MIMEImage
from dotenv import load_dotenv

load_dotenv(override=True)

# --- MULTI-ACCOUNT CONFIGURATION ---
# Gmail limit for free accounts updated to 496 per account/day.
ACCOUNTS = [
    {
        "email": os.getenv('SENDER_EMAIL_1', 'account1@gmail.com'),
        "password": os.getenv('SMTP_PASSWORD_1', 'app_pass_1'),
        "sender_name": "AI Catalyst Team",
        "smtp_server": "smtp.gmail.com",
        "smtp_port": 587,
        "daily_limit": 496
    },
    {
        "email": os.getenv('SENDER_EMAIL_2', 'account2@gmail.com'),
        "password": os.getenv('SMTP_PASSWORD_2', 'app_pass_2'),
        "sender_name": "AI Catalyst Team",
        "smtp_server": "smtp.gmail.com",
        "smtp_port": 587,
        "daily_limit": 496
    },
    {
        "email": os.getenv('SENDER_EMAIL_3', 'account3@gmail.com'),
        "password": os.getenv('SMTP_PASSWORD_3', 'app_pass_3'),
        "sender_name": "AI Catalyst Team",
        "smtp_server": "smtp.gmail.com",
        "smtp_port": 587,
        "daily_limit": 496
    },
    {
        "email": os.getenv('SENDER_EMAIL_4', 'account4@gmail.com'),
        "password": os.getenv('SMTP_PASSWORD_4', 'app_pass_4'),
        "sender_name": "AI Catalyst Team",
        "smtp_server": "smtp.gmail.com",
        "smtp_port": 587,
        "daily_limit": 496
    },
    {
        "email": os.getenv('SENDER_EMAIL_5', 'account5@gmail.com'),
        "password": os.getenv('SMTP_PASSWORD_5', 'app_pass_5'),
        "sender_name": "AI Catalyst Team",
        "smtp_server": "smtp.gmail.com",
        "smtp_port": 587,
        "daily_limit": 496
    },
    # Add additional accounts here if needed
]

SHEET_FILENAME = "guests_data.csv"
QR_FOLDER = "QR code data"
DELAY_BETWEEN_EMAILS = 1.0

EMAIL_SUBJECT = "Your Official Ticket & QR Code for the Event"
EMAIL_TEMPLATE_HTML = """
<!DOCTYPE html>
<html>
<body style="font-family: Arial, sans-serif; color: #e6e6f0; line-height: 1.6; margin: 0; padding: 0; background-color: #14101f;">
    <div style="max-width: 600px; margin: 20px auto; padding: 0; border-radius: 14px; overflow: hidden; background: linear-gradient(180deg, #1a1530 0%, #3a1d5c 100%); border: 1px solid #3a2b5c;">

        <!-- Body -->
        <div style="padding: 36px 30px 30px 30px;">
            <p style="color: #d8d3e8; font-size: 1.05rem; margin-top: 0;">Welcome to AI Catalyst V.04! 💜</p>
            <p style="color: #d8d3e8;">We're really happy to have you with us and can't wait to see you there.</p>
            <p style="color: #d8d3e8;">Here's the QR code for the event details.</p>

            <div style="text-align: center; margin: 30px 0; padding: 20px; background: rgba(255,255,255,0.05); border: 1px solid #4a3670; border-radius: 12px;">
                <img src="cid:qrcode_img" alt="Your QR Code" style="width: 220px; height: 220px; border: 2px solid #6b4fa0; padding: 6px; background: #fff; border-radius: 8px;">
                <p style="font-size: 0.85rem; color: #b3a8cc; margin-top: 10px; font-family: monospace;">Ticket ID: {uuid}</p>
            </div>

            <p style="color: #d8d3e8;">See you soon!</p>

            <p style="color: #d8d3e8;">If you have any issues or questions, reply directly to this email.</p>
        </div>

        <!-- Footer -->
        <div style="text-align: center; padding: 16px; background-color: #14101f; color: #8a7fa8; font-size: 0.75rem; letter-spacing: 1px;">
            AI CATALYST 2026 · FRAMEWORK
        </div>

    </div>
</body>
</html>
"""

In [13]:
local_file = Path(SHEET_FILENAME)
if not local_file.exists():
    raise FileNotFoundError(f"Cannot find '{SHEET_FILENAME}'. Please run the QR generation script first.")

df = pd.read_csv(local_file)

# Dynamic column discovery to match Google Sheet headers case-insensitively
name_col = next((col for col in df.columns if col.strip().lower() == 'name'), 'Name')
email_col = next((col for col in df.columns if col.strip().lower() == 'email'), 'Email')
uuid_col = next((col for col in df.columns if col.strip().lower() == 'uuid'), 'UUID')

if 'email_sent' not in df.columns:
    df['email_sent'] = False

df['email_sent'] = df['email_sent'].fillna(False).astype(bool)

unsent_count = len(df[~df['email_sent']])
sent_count = len(df[df['email_sent']])

print(f"📊 Total Attendees: {len(df)}")
print(f"✅ Emails Already Sent: {sent_count}")
print(f"⏳ Remaining to Send: {unsent_count}")

📊 Total Attendees: 2795
✅ Emails Already Sent: 2791
⏳ Remaining to Send: 4


In [14]:
def create_smtp_connection(acc):
    """Establishes an active, authenticated SMTP connection safely."""
    port = int(acc['smtp_port'])
    host = acc['smtp_server']
    
    if port == 465:
        server = smtplib.SMTP_SSL(host, port)
    else:
        server = smtplib.SMTP(host, port)
        server.starttls()
        
    server.login(acc['email'], acc['password'])
    return server

def send_qr_email(smtp_conn, sender_acc, recipient_email, recipient_name, uuid_val):
    qr_file_path = Path(QR_FOLDER) / f"{uuid_val}.png"
    
    if not qr_file_path.exists():
        print(f"⚠️ Missing QR code file for {recipient_name} ({uuid_val}). Skipping.")
        return False

    msg = MIMEMultipart('related')
    msg['Subject'] = EMAIL_SUBJECT
    msg['From'] = f"{sender_acc['sender_name']} <{sender_acc['email']}>"
    msg['To'] = recipient_email

    html_content = EMAIL_TEMPLATE_HTML.format(
        name=recipient_name,
        uuid=uuid_val,
        sender_name=sender_acc['sender_name']
    )
    
    msg.attach(MIMEText(html_content, 'html'))

    with open(qr_file_path, 'rb') as f:
        img = MIMEImage(f.read())
        img.add_header('Content-ID', '<qrcode_img>')
        img.add_header('Content-Disposition', 'inline', filename=f"{uuid_val}.png")
        msg.attach(img)

    smtp_conn.send_message(msg)
    return True

In [15]:
pending_df = df[~df['email_sent']]

if pending_df.empty:
    print("🎉 All emails have been sent! Nothing left to process.")
else:
    account_idx = 0
    current_acc = ACCOUNTS[account_idx]
    emails_sent_by_acc = 0
    server = None

    print(f"🚀 Starting dispatch across {len(ACCOUNTS)} sender accounts...\n")

    try:
        server = create_smtp_connection(current_acc)
        print(f"🔑 Connected using account: {current_acc['email']}")
    except Exception as e:
        print(f"❌ Initial connection failed for {current_acc['email']}: {e}")

    for idx, row in pending_df.iterrows():
        # Rotate account if 496 daily limit is reached
        if emails_sent_by_acc >= current_acc.get('daily_limit', 496):
            print(f"\n✋ Reached safety limit ({emails_sent_by_acc}) for {current_acc['email']}.")
            if server:
                try: server.quit() 
                except: pass

            account_idx += 1
            if account_idx >= len(ACCOUNTS):
                print("🚨 All accounts have reached their 496 daily limit! Stopping execution.")
                break

            current_acc = ACCOUNTS[account_idx]
            emails_sent_by_acc = 0
            try:
                server = create_smtp_connection(current_acc)
                print(f"🔄 Switched to account: {current_acc['email']}")
            except Exception as e:
                print(f"❌ Failed connecting to next account {current_acc['email']}: {e}")
                break

        guest_name = str(row[name_col]).strip()
        guest_email = str(row[email_col]).strip()
        guest_uuid = str(row[uuid_col]).strip()

        try:
            success = send_qr_email(server, current_acc, guest_email, guest_name, guest_uuid)
            if success:
                df.at[idx, 'email_sent'] = True
                emails_sent_by_acc += 1
                
                print(f"✅ Sent to {guest_name} <{guest_email}> via {current_acc['email']}")
                df.to_csv(SHEET_FILENAME, index=False)
                
            time.sleep(DELAY_BETWEEN_EMAILS)

        except (smtplib.SMTPDataError, smtplib.SMTPResponseException) as quota_err:
            print(f"⚠️ Quota/Rate limit error on {current_acc['email']}: {quota_err}")
            # Force account rotation on quota error
            emails_sent_by_acc = current_acc.get('daily_limit', 496)
            
        except Exception as e:
            print(f"❌ Failed sending to {guest_name} ({guest_email}): {e}")

    if server:
        try: server.quit()
        except: pass

    print(f"\n✨ Batch complete. Progress saved to {SHEET_FILENAME}.")

🚀 Starting dispatch across 5 sender accounts...

🔑 Connected using account: seif.mlashin1911@gmail.com
✅ Sent to ahmed elsissy <ahmedelsissy789@gmail.com> via seif.mlashin1911@gmail.com
✅ Sent to Ali Ahmed <20240337@stud.fci-cu.edu.eg> via seif.mlashin1911@gmail.com
✅ Sent to Rokkia farhan <Rokkia.2025023477@bua.edu.eg> via seif.mlashin1911@gmail.com
✅ Sent to Omar Elsayed Aboamer <3omar.elsayed332@gmail.com> via seif.mlashin1911@gmail.com

✨ Batch complete. Progress saved to guests_data.csv.
